In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_json("data/llm_hallucination_detection_dataset.jsonl", lines=True)
unique = df.drop_duplicates(["prompt", "context", "response"]).copy()
train_df, test_df = train_test_split(unique, test_size=0.2, random_state=42)
print(len(train_df), len(test_df))

153 39


In [2]:
import re

def overlap(response, context):
    response_words = set(re.findall(r"[a-z0-9]+", str(response).lower()))
    context_words = set(re.findall(r"[a-z0-9]+", str(context).lower()))
    if not response_words:
        return 0
    return len(response_words & context_words) / len(response_words)

word_pred = test_df.apply(
    lambda row: "hallucinated" if overlap(row["response"], row["context"]) < 0.6 else "faithful",
    axis=1,
)
print((word_pred == test_df["label"]).mean())

0.9743589743589743


In [3]:
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

label2id = {"faithful": 0, "hallucinated": 1}
id2label = {0: "faithful", 1: "hallucinated"}

class HallucinationDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=256):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        hypothesis = str(row["prompt"]) + " " + str(row["response"])
        encoded = self.tokenizer(
            str(row["context"]),
            hypothesis,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item["labels"] = torch.tensor(label2id[row["label"]])
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {"accuracy": accuracy_score(labels, preds), "weighted_f1": f1}

def train_and_score(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, id2label=id2label, label2id=label2id
    )
    args = TrainingArguments(
        output_dir=f"outputs/{model_name}",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        eval_strategy="no",
        save_strategy="no",
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=HallucinationDataset(train_df, tokenizer),
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    predicted = trainer.predict(HallucinationDataset(test_df, tokenizer))
    y_pred = [id2label[i] for i in predicted.predictions.argmax(axis=-1)]
    print(model_name)
    print(classification_report(test_df["label"], y_pred, digits=4))

C:\Users\Sh_al\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train_and_score("bert-base-uncased")

C:\Users\Sh_al\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sh_al\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Sh_al\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_dow

Step,Training Loss


bert-base-uncased
              precision    recall  f1-score   support

    faithful     0.9412    0.8000    0.8649        20
hallucinated     0.8182    0.9474    0.8780        19

    accuracy                         0.8718        39
   macro avg     0.8797    0.8737    0.8715        39
weighted avg     0.8813    0.8718    0.8713        39



In [5]:
train_and_score("roberta-base")

C:\Users\Sh_al\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sh_al\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 579.34it/s]
[transformers] RobertaForSequenceC

Step,Training Loss


roberta-base
              precision    recall  f1-score   support

    faithful     0.9412    0.8000    0.8649        20
hallucinated     0.8182    0.9474    0.8780        19

    accuracy                         0.8718        39
   macro avg     0.8797    0.8737    0.8715        39
weighted avg     0.8813    0.8718    0.8713        39



## Result on the 39 unique test rows

Word check: accuracy 0.974.
BERT-base: accuracy 0.872.
RoBERTa-base: accuracy 0.872.

Both models miss the same number of hallucinated answers as the word check, and they also mark some faithful answers as hallucinated. The word check stays the current method.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

text_train = train_df["prompt"] + " " + train_df["response"]
text_test = test_df["prompt"] + " " + test_df["response"]

model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
model.fit(text_train, train_df["label"])
print(classification_report(test_df["label"], model.predict(text_test), digits=4))

              precision    recall  f1-score   support

    faithful     0.9375    0.7500    0.8333        20
hallucinated     0.7826    0.9474    0.8571        19

    accuracy                         0.8462        39
   macro avg     0.8601    0.8487    0.8452        39
weighted avg     0.8620    0.8462    0.8449        39



## No context

A logistic regression model trained on the question and the answer only, with the context left out, scores 0.846 accuracy on the 39 test rows. The word check scores 0.974 and needs the context. BERT and RoBERTa with the context score 0.872.